# Ecommerce Mini Warehouse — ETL & Quality Checks

This notebook orchestrates the full ETL pipeline from raw CSV files into a
PostgreSQL star-schema data warehouse and then runs data quality checks.

**Stack:** Python · pandas · SQLAlchemy · psycopg2 · PostgreSQL

## Workflow
1. Connect to `ecommerce_warehouse` PostgreSQL database
2. Run DDL (`01_schema_ddl.sql`) — create dimension & fact tables
3. Run staging DDL (`02_staging_tables.sql`) — create raw landing tables
4. Load CSV files into staging tables
5. Run ETL (`03_etl_insert_dim_fact.sql`) — transform and load dims/facts
6. Run quality checks (`04_quality_checks.sql`) and display results

## 0. Imports & Configuration

In [ ]:
import os
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

# ── Connection settings ──────────────────────────────────────────────────────
DB_USER     = os.getenv('PG_USER',     'postgres')
DB_PASSWORD = os.getenv('PG_PASSWORD', 'postgres')
DB_HOST     = os.getenv('PG_HOST',     'localhost')
DB_PORT     = os.getenv('PG_PORT',     '5432')
DB_NAME     = os.getenv('PG_DB',       'ecommerce_warehouse')

CONNECTION_STRING = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# ── File paths ───────────────────────────────────────────────────────────────
BASE_DIR    = Path('..').resolve()
DATA_DIR    = BASE_DIR / 'data'
SQL_DIR     = BASE_DIR / 'sql'

print('Base directory :', BASE_DIR)
print('Data directory :', DATA_DIR)
print('SQL  directory :', SQL_DIR)

## 1. Database Connection

In [ ]:
engine = create_engine(CONNECTION_STRING, future=True)

with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('Connected:', result.fetchone()[0])

## 2. Helper Functions

In [ ]:
def run_sql_file(filepath: Path, engine) -> None:
    """Execute a .sql file, skipping psql meta-commands (\\COPY, \\i, etc.)."""
    raw = filepath.read_text(encoding='utf-8')
    # Strip psql backslash meta-commands which cannot run via the PostgreSQL
    # server wire protocol (they are handled by the psql client only).
    sql = '\n'.join(
        line for line in raw.splitlines()
        if not line.lstrip().startswith('\\')
    )
    with engine.begin() as conn:
        conn.execute(text(sql))
    print(f'\u2713 Executed: {filepath.name}')


def query_to_df(sql: str, engine) -> pd.DataFrame:
    """Run a SELECT statement and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


## 3. Run DDL — Create Schema

In [ ]:
run_sql_file(SQL_DIR / '01_schema_ddl.sql', engine)

## 4. Create Staging Tables

In [ ]:
run_sql_file(SQL_DIR / '02_staging_tables.sql', engine)

## 5. Load CSVs → Staging

In [ ]:
csv_to_table = {
    'sales_raw.csv'  : 'stg_sales',
    'churn_raw.csv'  : 'stg_churn',
    'ab_test_raw.csv': 'stg_ab_test',
}

for filename, table in csv_to_table.items():
    df = pd.read_csv(DATA_DIR / filename, dtype=str).fillna('')
    # rename 'group' column to avoid reserved-word clash
    if 'group' in df.columns:
        df = df.rename(columns={'group': 'ab_group'})
    df.to_sql(table, engine, schema='staging', if_exists='append', index=False)
    print(f'\u2713 Loaded {len(df):,} rows \u2192 staging.{table}')


In [ ]:
# Quick preview of each staging table
for table in ['stg_sales', 'stg_churn', 'stg_ab_test']:
    df = query_to_df(f'SELECT * FROM staging.{table} LIMIT 3', engine)
    print(f'\n\u2500\u2500 {table} \u2500\u2500')
    display(df)


## 6. Run ETL — Load Dimensions & Facts

In [ ]:
run_sql_file(SQL_DIR / '03_etl_insert_dim_fact.sql', engine)

In [ ]:
# Row counts after ETL
tables = [
    'dw.dim_customer', 'dw.dim_product', 'dw.dim_date', 'dw.dim_channel',
    'dw.fact_sales', 'dw.fact_churn', 'dw.fact_ab_test'
]
count_sql = ' UNION ALL '.join(
    f"SELECT '{t}' AS table_name, COUNT(*) AS row_count FROM {t}"
    for t in tables
)
counts_df = query_to_df(count_sql + ' ORDER BY table_name', engine)
display(counts_df)


## 7. Data Quality Checks

In [ ]:
# ── Referential integrity ────────────────────────────────────────────────────
ri_checks = [
    ('fact_sales → dim_customer',
     'SELECT COUNT(*) AS issues FROM dw.fact_sales fs '
     'WHERE NOT EXISTS (SELECT 1 FROM dw.dim_customer dc WHERE dc.customer_id = fs.customer_id)'),
    ('fact_sales → dim_product',
     'SELECT COUNT(*) AS issues FROM dw.fact_sales fs '
     'WHERE NOT EXISTS (SELECT 1 FROM dw.dim_product dp WHERE dp.product_id = fs.product_id)'),
    ('fact_sales → dim_date',
     'SELECT COUNT(*) AS issues FROM dw.fact_sales fs '
     'WHERE NOT EXISTS (SELECT 1 FROM dw.dim_date dd WHERE dd.date_key = fs.date_key)'),
    ('fact_churn → dim_customer',
     'SELECT COUNT(*) AS issues FROM dw.fact_churn fc '
     'WHERE NOT EXISTS (SELECT 1 FROM dw.dim_customer dc WHERE dc.customer_id = fc.customer_id)'),
    ('fact_ab_test → dim_customer',
     'SELECT COUNT(*) AS issues FROM dw.fact_ab_test fa '
     'WHERE NOT EXISTS (SELECT 1 FROM dw.dim_customer dc WHERE dc.customer_id = fa.user_id)'),
]

ri_results = []
for name, sql in ri_checks:
    df = query_to_df(sql, engine)
    ri_results.append({'check': name, 'issues': int(df['issues'].iloc[0])})

ri_df = pd.DataFrame(ri_results)
ri_df['status'] = ri_df['issues'].apply(lambda x: '\u2713 PASS' if x == 0 else '\u2717 FAIL')
display(ri_df)


In [ ]:
# ── Business rule checks ─────────────────────────────────────────────────────
biz_checks = [
    ('fact_sales: negative revenue',
     'SELECT COUNT(*) AS issues FROM dw.fact_sales WHERE revenue < 0'),
    ('fact_sales: zero/negative quantity',
     'SELECT COUNT(*) AS issues FROM dw.fact_sales WHERE quantity <= 0'),
    ('fact_churn: churn_date set but churn_flag=false',
     'SELECT COUNT(*) AS issues FROM dw.fact_churn WHERE churn_flag = FALSE AND churn_date IS NOT NULL'),
    ('fact_ab_test: invalid group_name',
     "SELECT COUNT(*) AS issues FROM dw.fact_ab_test WHERE group_name NOT IN ('control','treatment')"),
    ('dim_customer: invalid income_band',
     "SELECT COUNT(*) AS issues FROM dw.dim_customer WHERE income_band NOT IN ('Low','Medium','High','Unknown')"),
]

biz_results = []
for name, sql in biz_checks:
    df = query_to_df(sql, engine)
    biz_results.append({'check': name, 'issues': int(df['issues'].iloc[0])})

biz_df = pd.DataFrame(biz_results)
biz_df['status'] = biz_df['issues'].apply(lambda x: '\u2713 PASS' if x == 0 else '\u2717 FAIL')
display(biz_df)


## 8. Exploratory Analytics

In [ ]:
# Revenue by channel
revenue_by_channel = query_to_df("""
    SELECT
        dc.channel_name,
        COUNT(*)                   AS num_sales,
        SUM(fs.revenue)            AS total_revenue,
        ROUND(AVG(fs.revenue), 2)  AS avg_order_revenue
    FROM dw.fact_sales fs
    JOIN dw.dim_channel dc USING (channel_id)
    GROUP BY dc.channel_name
    ORDER BY total_revenue DESC
""", engine)
display(revenue_by_channel)


In [ ]:
# Churn rate by region
churn_by_region = query_to_df("""
    SELECT
        cust.region,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN fc.churn_flag THEN 1 ELSE 0 END) AS churned,
        ROUND(
            100.0 * SUM(CASE WHEN fc.churn_flag THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS churn_rate_pct
    FROM dw.fact_churn fc
    JOIN dw.dim_customer cust USING (customer_id)
    GROUP BY cust.region
    ORDER BY churn_rate_pct DESC
""", engine)
display(churn_by_region)


In [ ]:
# A/B test conversion rates
ab_results = query_to_df("""
    SELECT
        de.experiment_name,
        fat.group_name,
        COUNT(*) AS participants,
        SUM(CASE WHEN fat.converted THEN 1 ELSE 0 END) AS conversions,
        ROUND(
            100.0 * SUM(CASE WHEN fat.converted THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS conversion_rate_pct,
        ROUND(SUM(fat.revenue), 2) AS total_revenue
    FROM dw.fact_ab_test fat
    JOIN dw.dim_experiment de USING (experiment_id)
    GROUP BY de.experiment_name, fat.group_name
    ORDER BY de.experiment_name, fat.group_name
""", engine)
display(ab_results)


In [ ]:
# Monthly revenue trend
monthly_revenue = query_to_df("""
    SELECT
        dd.year,
        dd.month,
        COUNT(*)             AS num_orders,
        SUM(fs.revenue)      AS total_revenue,
        SUM(fs.gross_profit) AS total_gross_profit
    FROM dw.fact_sales fs
    JOIN dw.dim_date dd ON dd.date_key = fs.date_key
    GROUP BY dd.year, dd.month
    ORDER BY dd.year, dd.month
""", engine)
display(monthly_revenue)
